# Actividad 8 – CNN (TensorFlow / Keras) y **Transfer Learning** (Google Colab)

**REA 1 · Deep Learning** · Semana 8

**Objetivo del notebook:** mostrar un flujo completo de **clasificación de imágenes** (CIFAR-10) con
1) una **CNN entrenable desde cero** y
2) un modelo con **base preentrenada** (ImageNet) y capas superiores entrenables.

**Evidencia:** partición de datos, entrenamiento, **loss/accuracy** en entrenamiento y validación, matriz de confusión, comparación y **conclusiones** (sección final).

## 0. Requisito de entorno (léelo si ejecutas en **tu PC**)

- **Google Colab (recomendado):** suele traer **TensorFlow** y **Python 3.10/3.11**; *Ejecutar todo* debería funcionar.
- **Windows con Python 3.13** (Store / instalación reciente): TensorFlow a menudo **no tiene wheel oficial** o queda la instalación **a medias** → error `No module named 'tensorflow.python'` al importar.

**Qué hacer en local (elige una):**
1. Usar el notebook en **Colab** (cero conflicto con la versión de Python del sistema).
2. Crear un **entorno virtual** con **Python 3.11 o 3.12** (desde [python.org](https://www.python.org/downloads/) o con `py -3.12` si tienes el launcher) e instalar: `pip install tensorflow scikit-learn matplotlib seaborn jupyter`.
3. Reinstalar TensorFlow: `pip uninstall tensorflow tensorflow-intel -y` y luego `pip install "tensorflow>=2.15"` (solo si tu Python es **&lt; 3.13** o ya hay soporte en PyPI para 3.13 en tu plataforma).

Si aún falla, la celda de imports siguiente mostrará un mensaje detallado.

In [ ]:
# En Colab, si hace falta, descomenta:  !pip install -q "tensorflow>=2.15" scikit-learn matplotlib seaborn
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

vi = sys.version_info
print("Python:", sys.version.split()[0], "(major.minor =", f"{vi.major}.{vi.minor})")
if vi >= (3, 13):
    print(
        "\n[AVISO] Con Python 3.13+ TensorFlow a menudo falla o queda mal instalado en Windows. "
        "Recomendacion: usar Google Colab, o un venv con Python 3.11/3.12.\n"
    )

import numpy as np
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, optimizers, callbacks
except Exception as e:
    print("\n========== Error al importar TensorFlow ==========\n", repr(e), "\n")
    print(
        "Pasos sugeridos:\n"
        "  1) Ejecutar en Google Colab (sube este .ipynb).\n"
        "  2) En Windows: venv con Python 3.11/3.12 y pip install tensorflow.\n"
        "  3) Reinstalar: pip uninstall tensorflow tensorflow-intel -y  &&  pip install tensorflow\n"
    )
    raise
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


## 1. Datos: CIFAR-10 (carga, normalización, subconjunto *limitado*)

Se usa CIFAR-10 (32×32, RGB, 10 clases). Para aproximar un **contexto con datos localmente restringidos**, se toma un **subconjunto fijo** del conjunto de entrenamiento para **ambos** modelos (misma partición, misma dificultad de comparación).

In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_all, y_test = y_all.ravel(), y_test.ravel()
class_names = [
    "avión", "coche", "pájaro", "gato", "venado", "perro", "rana", "caballo", "barco", "camión"
]

# Mismo límite para scratch y transfer (comparación controlada)
N_LIMIT = 8000
x_part = x_all[:N_LIMIT].astype("float32")
y_part = y_all[:N_LIMIT]

# Validación: 20% del subconjunto
x_train, x_val, y_train, y_val = train_test_split(
    x_part, y_part, test_size=0.2, random_state=SEED, stratify=y_part
)
# x_test se deja al final para evaluación (test fijo, no usado en ajuste de hiperparámetros)
x_test = x_test.astype("float32")

# CNN desde cero: entradas en [0, 1]
X_train_01 = x_train / 255.0
X_val_01 = x_val / 255.0
X_test_01 = x_test / 255.0
NUM_CLASS = 10
INPUT_SHAPE = (32, 32, 3)
print("Train:", X_train_01.shape, "| Val:", X_val_01.shape, "| Test:", X_test_01.shape)

## 2. Modelo A: CNN *from scratch* (entrenar todos los parámetros)

Arquitectura pequeña y típica: bloques `Conv2D` + `MaxPool`, `GlobalAveragePooling2D` y `Dense` de salida.

In [ ]:
def build_cnn_scratch():
    m = models.Sequential(
        [
            layers.Input(shape=INPUT_SHAPE),
            layers.Conv2D(32, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(2),
            layers.Conv2D(64, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(2),
            layers.Conv2D(64, 3, padding="same", activation="relu"),
            layers.GlobalAveragePooling2D(),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.2),
            layers.Dense(NUM_CLASS, activation="softmax", name="probs"),
        ],
        name="cnn_scratch_cifar10",
    )
    m.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return m


model_scratch = build_cnn_scratch()
model_scratch.summary()

In [ ]:
EPOCHS = 30
BATCH = 64

es = callbacks.EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1
)
rlr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

history_scratch = model_scratch.fit(
    X_train_01,
    y_train,
    batch_size=BATCH,
    epochs=EPOCHS,
    validation_data=(X_val_01, y_val),
    callbacks=[es, rlr],
    verbose=1,
)

In [ ]:
def plot_history(hist, title):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(hist.history["loss"], label="entrenamiento")
    ax[0].plot(hist.history["val_loss"], label="validación")
    ax[0].set_title("Pérdida (loss)")
    ax[0].set_xlabel("Época")
    ax[0].legend()
    ax[1].plot(hist.history["accuracy"], label="entrenamiento")
    ax[1].plot(hist.history["val_accuracy"], label="validación")
    ax[1].set_title("Exactitud (accuracy)")
    ax[1].set_xlabel("Época")
    ax[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_history(history_scratch, "Modelo A – CNN desde cero")

In [ ]:
# Evaluación: test
loss_s, acc_s = model_scratch.evaluate(X_test_01, y_test, verbose=0)
print(f"[Scratch] test loss: {loss_s:.4f}  |  test accuracy: {acc_s:.4f}")
y_pred_s = np.argmax(model_scratch.predict(X_test_01, verbose=0), axis=1)

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASS))
    plt.figure(figsize=(6, 4.5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title(title)
    plt.ylabel("Verdadero")
    plt.xlabel("Predicho")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


plot_cm(y_test, y_pred_s, "Matriz de confusión – CNN desde cero (test)")
print(classification_report(y_test, y_pred_s, target_names=class_names, digits=3))

## 3. Modelo B: **Transfer learning** (MobileNetV2 preentrenado en ImageNet)

- La base convolucional conserva **pesos de ImageNet** y permanece **congelada** (no entrenable) en esta fase, para aprovechar **representaciones** ya útiles.
- Se añade un cabezal de **clasificación** para 10 clases. La preprocesamiento de MobileNetV2 requiere `preprocess_input` (escala distinta a 0–1).

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Mismo split numérico; preprocesamiento específico del modelo
X_train_m = preprocess_input(x_train)
X_val_m = preprocess_input(x_val)
X_test_m = preprocess_input(x_test)

base = MobileNetV2(
    input_shape=INPUT_SHAPE, include_top=False, weights="imagenet", pooling=None
)
base.trainable = False  # fase: solo cabezal

_inputs = base.input
x = base.output
x = layers.GlobalAveragePooling2D(name="avg_pool")(x)
x = layers.Dropout(0.3)(x)
_outputs = layers.Dense(NUM_CLASS, activation="softmax", name="probs_tl")(x)
model_tl = models.Model(_inputs, _outputs, name="mobilenetv2_head_frozen")

model_tl.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
model_tl.summary()

In [ ]:
es2 = callbacks.EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1
)
rlr2 = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)
history_tl = model_tl.fit(
    X_train_m,
    y_train,
    batch_size=BATCH,
    epochs=EPOCHS,
    validation_data=(X_val_m, y_val),
    callbacks=[es2, rlr2],
    verbose=1,
)

In [ ]:
plot_history(history_tl, "Modelo B – Transfer learning (base congelada)")

loss_t, acc_t = model_tl.evaluate(X_test_m, y_test, verbose=0)
print(f"[Transfer] test loss: {loss_t:.4f}  |  test accuracy: {acc_t:.4f}")
y_pred_t = np.argmax(model_tl.predict(X_test_m, verbose=0), axis=1)

In [ ]:
plot_cm(y_test, y_pred_t, "Matriz de confusión – Transfer learning (test)")
print(classification_report(y_test, y_pred_t, target_names=class_names, digits=3))

## 4. Comparación (mismo subconjunto, misma val split, test común)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.2))
mets = ["Test accuracy", "Test loss (×10 para gráficar)"]
vals_acc = [acc_s, acc_t]
xpos = np.arange(2)
ax.bar(xpos - 0.2, [acc_s, acc_t], 0.4, label="accuracy", color="steelblue")
ax.set_xticks(xpos)
ax.set_xticklabels(["CNN scratch", "Transfer (MobileNetV2)"])
ax.set_ylabel("Exactitud (test)")
ax.set_ylim(0, 1.05)
ax.legend()
for i, v in enumerate(vals_acc):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
ax.set_title("Comparación en test (mismos datos de entrenamiento/validación)")
plt.tight_layout()
plt.show()

print("Resumen test:")
print(f"  CNN scratch      — loss: {loss_s:.4f}  acc: {acc_s:.4f}")
print(f"  Transfer learning— loss: {loss_t:.4f}  acc: {acc_t:.4f}")

## 5. Análisis (rúbrica)

- **Convergencia:** con `EarlyStopping` y `ReduceLROnPlateau` se busca un entrenamiento estable sin fijar épocas excesivas.
- **Trade-off scratch vs. transfer:** con **pocas** muestras de entrenamiento, la **base de ImageNet** aporta filtros **ya útiles**; la CNN pequeña debe aprender bordes y texturas **desde cero**, lo que suele ser más duro con datos limitados.
- **CIFAR-10 a 32×32** no coincide con el espacio de resolución de ImageNet, por lo que la ventaja de transfer no es “gratis”: aun así se observa a menudo **mejora relativa** o al menos mejores curvas de validación *en este pipeline fijo*.

> Si el *transfer* no superara al scratch en un run concreto, se documentaría: variación estocástica, épocas, learning rate, o conveniencia de afinar (*fine-tune*) la parte final de la base—fuera de alcance mínimo de esta actividad.

## 6. Conclusiones (3–5 puntos)

1. Se implementó un **pipeline** reproducible: carga, partición, normalización, dos arquitecturas, entrenamiento y evaluación con **pérdida, accuracy** y **matriz de confusión** en *test*.

2. La **CNN desde cero** ofrece control total y baja carga de modelo descargable, a costa de **más dificultad** para aprovechar un dataset reducido.

3. El **modelo con MobileNetV2** congela la base e inserta un cabezal denso: aprovecha **pesos preentrenados** coherente con *transfer learning* en datos limitados.

4. **Comparación controlada:** mismas muestras de entrenamiento/validación y el mismo *test*; la métrica principal **accuracy** en *test* permite juzgar el desempeño relativo bajo un mismo criterio.

5. **Recomendación práctica:** con datos escasos, probar **transfer** + ajuste fino parcial; con datos amplios y tareas distintas de ImageNet, valorar entrenar más capas o una CNN dedicada. **Trade-off:** *transfer* añade dependencia de modelo y preprocesado; *scratch* es más simple pero puede converger peor.